# Comparison: 7 compact tabular baselines (seed 163)

The training code is `train_baseline.py` (one model per run; compact PyTorch adaptations sized for 21D features). This notebook trains the 7 models in 7 processes and aggregates the results.

- Data: `../data/features_21d` (21 features, 4 classes; the StandardScaler is fitted on train only), same splits and seed 163 as Proposed.
- Protocol: 100 epochs, batch 2048, AdamW (lr 1e-3, wd 1e-4; NODE lr 2e-3), CosineAnnealingLR, cross-entropy.
- Model selection: keep the epoch with the best validation accuracy; the 62,002-window test set is evaluated once with those weights.
- Model parameter counts are listed below:

| Model | Parameters |
|---|---:|
| TabNet | 1,076 |
| FT-Transformer | 4,044 |
| AutoInt | 4,658 |
| DCN-V2 | 3,997 |
| ExcelFormer | 16,100 |
| Tab-ResNet | 18,948 |
| NODE | 2,404 |

Artifacts: `results/Experiment_<model>/` (best weights, training_metrics.csv, test_predictions, summary.json) and `logs/<model>.log`.
The released weights and results are in `checkpoints/` and `released_results/`; `evaluate_checkpoints.ipynb` does no training and recomputes the test results from the weights.

In [ ]:
# 7 processes in parallel, one model each; stdout goes to logs/<model>.log
import subprocess, sys, time
from pathlib import Path

NAMES = ['TabNet', 'FT-Transformer', 'AutoInt', 'DCN-V2', 'ExcelFormer', 'Tab-ResNet', 'NODE']
PAR = Path('.').resolve()
LOG = PAR / 'logs'
LOG.mkdir(parents=True, exist_ok=True)
WORKER = Path('train_baseline.py').resolve()

procs, handles = {}, {}
for n in NAMES:
    h = open(LOG / (n.replace('-', '') + '.log'), 'w')
    p = subprocess.Popen([sys.executable, str(WORKER), n],
                         stdout=h, stderr=subprocess.STDOUT, cwd=str(PAR))
    procs[n], handles[n] = p, h
print('[t=000s] launched %d processes, logs -> %s' % (len(procs), LOG), flush=True)

t0 = time.time()
while True:
    alive = [n for n in NAMES if procs[n].poll() is None]
    done = [n for n in NAMES if n not in alive]
    print('[t=%03.0fs] done %d/%d  finished: %-45s running: %s'
          % (time.time() - t0, len(done), len(NAMES), ','.join(done) or '-', ','.join(alive) or '-'),
          flush=True)
    if not alive:
        break
    time.sleep(30)

for n in NAMES:
    handles[n].close()
    print('%-16s returncode = %d' % (n, procs[n].returncode))
print('total wall = %.1f s (%.1f min)' % (time.time() - t0, (time.time() - t0) / 60.0))

In [ ]:
# aggregate
import json, os
from pathlib import Path

RES = Path(os.environ.get('OUT_DIR', 'results'))
rows = []
for d in sorted(RES.glob('Experiment_*')):
    f = d / 'summary.json'
    if f.exists():
        rows.append(json.loads(f.read_text(encoding='utf-8')))
rows.sort(key=lambda r: -r['test_acc_percent'])

print('%-16s %8s %10s %10s %10s %8s %8s' %
      ('model', 'params', 'train_acc', 'best_val', 'test_acc', 'correct', 'sec'))
for r in rows:
    print('%-16s %8d %9.4f%% %9.4f%% %9.4f%% %8d %8.0f' %
          (r['model'], r['params'], r['train_acc_eval'] * 100, r['best_val_acc'] * 100,
           r['test_acc_percent'], r['test_correct'], r['seconds']))